In [ ]:
#0 Load Libraries and Configurations

import os
import wrds
import pandas as pd
import numpy as np

# ---------- User-configurable paths ----------
PATH_DATA_INTERMEDIATE = "/Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate"  # <-- change this
os.makedirs(PATH_DATA_INTERMEDIATE, exist_ok=True)

OUT_PARQUET = os.path.join(PATH_DATA_INTERMEDIATE, "monthlyFF.parquet")
OUT_CSV     = os.path.join(PATH_DATA_INTERMEDIATE, "monthlyFF.csv")

In [ ]:
#1 Load FF Data

SQL = """
SELECT
    date,
    mktrf,
    smb,
    hml,
    rf,
    umd
FROM ff.factors_monthly
WHERE date >= DATE '2000-01-01';
"""

In [ ]:
#2 FF Data Extraction From WRDS

db = wrds.Connection()
df = db.raw_sql(SQL, date_cols=["date"])

In [ ]:
#3 Data Cleaning

# -------- monthly index and cleanup --------
df["time_avail_m"] = df["date"].dt.to_period("M").dt.to_timestamp("MS")
df = df.drop(columns=["date"])

# (optional) enforce numeric dtypes
for col in ["mktrf", "smb", "hml", "rf", "umd"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# -------- save --------
df.to_parquet(OUT_PARQUET, index=False)
df.to_csv(OUT_CSV, index=False)

print("Saved:")
print(" -", OUT_PARQUET)
print(" -", OUT_CSV)
print(df.head())